# Pivot und Unpivot

[Referenz](https:\learn.microsoft.com\de-de\sql\t-sql\queries\from-using-pivot-and-unpivot?view=sql-server-ver16)

## Pivotiere die Umsätze der vier Oberkategorien (in Zeilen) für die Kalenderjahre 2014 – 2018 (in Spalten)

![pu001](Screenshots/pu001.png)

In [ ]:
WITH Umsatzdaten AS
(
  SELECT
    ober.ProduktKategorieID AS OberKategorieID,
    ober.Bezeichnung        AS OberKategorie,
    YEAR(b.Bestelldatum)  AS Jahr,
    CAST(bd.Bestellmenge * bd.Einzelpreis * (1 - bd.Rabatt) AS money) AS Umsatz
  FROM Dim.ProduktKategorie AS unter
  JOIN Dim.ProduktKategorie AS ober ON unter.OberKategorieID = ober.ProduktKategorieID
  JOIN Stamm.Produkt AS p ON unter.ProduktKategorieID = p.ProduktKategorieID
  JOIN Verkauf.BestellDetail AS bd ON p.ProduktID = bd.ProduktID
  JOIN Verkauf.Bestellung AS b ON bd.BestellungID = b.BestellungID
)
SELECT
  [OberKategorie],
  [2014],
  [2015],
  [2016],
  [2017],
  [2018]
FROM Umsatzdaten
PIVOT
(
  SUM(Umsatz) FOR Jahr IN ([2014],[2015],[2016],[2017],[2018])
) AS pt
ORDER BY [OberKategorie];

(4 rows affected)

OberKategorie | 2014        | 2015        | 2016        | 2017        | 2018      
--------------+-------------+-------------+-------------+-------------+-----------
Accessories   | 28600,53    | 37290,25    | 37419,31    | 32272,56    | 33232,38  
Bikes         | 5182202,46  | 5616936,37  | 5400692,90  | 5827967,60  | 5772171,15
Clothing      | 68725,25    | 74021,99    | 62146,95    | 66372,46    | 69769,70  
Components    | 2279243,055 | 2392466,965 | 2279175,945 | 2530610,165 | 2539159,87
(4 rows)

Total execution time: 00:00:00.279

## Erstelle durch die vorherige Pivot-Aufgabe eine neue Tabelle PivotDaten1. Unpivotiere anschließend diese Tabelle per Abfrage.

![pu002](Screenshots/pu002.png)

In [10]:
-- SELECT ... INTO
DROP TABLE IF EXISTS PivotDaten1;
GO

WITH Umsatzdaten AS
(
  SELECT
    ober.ProduktKategorieID AS OberKategorieID,
    ober.Bezeichnung        AS OberKategorie,
    YEAR(b.Bestelldatum)  AS Jahr,
    CAST(bd.Bestellmenge * bd.Einzelpreis * (1 - bd.Rabatt) AS money) AS Umsatz
  FROM Dim.ProduktKategorie AS unter
  JOIN Dim.ProduktKategorie AS ober ON unter.OberKategorieID = ober.ProduktKategorieID
  JOIN Stamm.Produkt AS p ON unter.ProduktKategorieID = p.ProduktKategorieID
  JOIN Verkauf.BestellDetail AS bd ON p.ProduktID = bd.ProduktID
  JOIN Verkauf.Bestellung AS b ON bd.BestellungID = b.BestellungID
)
SELECT
  [OberKategorie],
  [2014],
  [2015],
  [2016],
  [2017],
  [2018]
INTO PivotDaten1
FROM Umsatzdaten
PIVOT
(
  SUM(Umsatz) FOR Jahr IN ([2014],[2015],[2016],[2017],[2018])
) AS pt

Commands completed successfully.

(4 rows affected)

Total execution time: 00:00:00.084

In [13]:
-- UNPIVOT
SELECT OberKategorie, Jahr, Umsatz
FROM PivotDaten1
UNPIVOT (Umsatz FOR Jahr IN ([2014],[2015],[2016],[2017],[2018])) AS unpvt;

(20 rows affected)

OberKategorie | Jahr | Umsatz     
--------------+------+------------
Components    | 2014 | 2279243,055
Components    | 2015 | 2392466,965
Components    | 2016 | 2279175,945
Components    | 2017 | 2530610,165
Components    | 2018 | 2539159,87 
Clothing      | 2014 | 68725,25   
Clothing      | 2015 | 74021,99   
Clothing      | 2016 | 62146,95   
Clothing      | 2017 | 66372,46   
Clothing      | 2018 | 69769,70   
Bikes         | 2014 | 5182202,46 
Bikes         | 2015 | 5616936,37 
Bikes         | 2016 | 5400692,90 
Bikes         | 2017 | 5827967,60 
Bikes         | 2018 | 5772171,15 
Accessories   | 2014 | 28600,53   
Accessories   | 2015 | 37290,25   
Accessories   | 2016 | 37419,31   
Accessories   | 2017 | 32272,56   
Accessories   | 2018 | 33232,38   
(20 rows)

Total execution time: 00:00:00.107

## Pivotiere die Umsätze der vier Oberkategorien (in Spalten) für die Kalenderjahre 2014 – 2018 (in Zeilen)

![pu003](Screenshots/pu003.png)

In [17]:
WITH umsatzdaten AS
(
    SELECT k.Bezeichnung AS [Oberkategorie]
    , YEAR(b.Bestelldatum) AS [Jahr]
    , d.Einzelpreis * d.Bestellmenge AS [Umsatz]
    FROM Dim.ProduktKategorie AS k
    JOIN Dim.ProduktKategorie AS uk
        ON k.ProduktKategorieID = uk.OberKategorieID
    JOIN Stamm.Produkt AS p
        ON uk.ProduktKategorieID = p.ProduktKategorieID
    JOIN Verkauf.BestellDetail AS d
        ON p.ProduktID = d.ProduktID
    JOIN Verkauf.Bestellung AS b
        ON d.BestellungID = b.BestellungID
        AND b.Bestelldatum >= '20140101'
        AND b.Bestelldatum < '20190101'
)
SELECT
    [Jahr],
    [Accessories],
    [Bikes],
    [Clothing],
    [Components]
FROM umsatzdaten
PIVOT(SUM(Umsatz) FOR Oberkategorie
    IN ([Accessories], [Bikes], [Clothing], [Components])
    ) AS pt
ORDER BY Jahr;

(5 rows affected)

Jahr | Accessories | Bikes      | Clothing | Components 
-----+-------------+------------+----------+------------
2014 | 28600,53    | 5182202,46 | 68725,25 | 2279243,055
2015 | 37290,25    | 5616936,37 | 74021,99 | 2392466,965
2016 | 37419,31    | 5400692,90 | 62146,95 | 2279175,945
2017 | 32272,56    | 5827967,60 | 66372,46 | 2530610,165
2018 | 33232,38    | 5772171,15 | 69769,70 | 2539159,87 
(5 rows)

Total execution time: 00:00:00.076

## Pivotiere den Umsatz der Mitarbeiter für die Geschäftsjahre 2014 – 2018 in Spalten, Mitarbeiter Personalnummer und Nachname in Zeilen.

![pu004](Screenshots/pu004.png)

In [16]:
WITH Unterabfrage AS
(
  SELECT
    m.MitarbeiterID,
    m.Vorname,
    m.Nachname,
    YEAR(b.bestelldatum) AS Jahr,
    (bd.Einzelpreis * bd.Bestellmenge * (1 - bd.Rabatt)) AS [Umsatz]
  FROM Personal.Mitarbeiter AS m
  LEFT JOIN Verkauf.Bestellung AS b ON m.MitarbeiterID = b.MitarbeiterID
  LEFT JOIN Verkauf.BestellDetail as bd ON b.BestellungID = bd.BestellungID
)
SELECT MitarbeiterID, Vorname, Nachname, [2014], [2015], [2016], [2017], [2018]
FROM Unterabfrage
PIVOT(SUM(Umsatz) FOR Jahr IN ([2014],[2015],[2016],[2017],[2018])) AS pv
Order By MitarbeiterID

(18 rows affected)

MitarbeiterID | Vorname  | Nachname  | 2014       | 2015       | 2016       | 2017       | 2018      
--------------+----------+-----------+------------+------------+------------+------------+-----------
1             | Nancy    | Davolio   | 173928,11  | 282147,635 | 277425,835 | 290017,96  | 304013,575
2             | Andrew   | Fuller    | 471795,40  | 632746,73  | 423345,545 | 579752,425 | 473277,28 
3             | Janet    | Leverling | 480291,45  | 528152,60  | 539140,575 | 490816,365 | 655688,93 
4             | Margaret | Peacock   | 468033,45  | 328754,20  | 475075,415 | 679597,48  | 600636,595
5             | Steven   | Buchanan  | 761494,45  | 641495,50  | 489579,945 | 614729,47  | 367222,10 
6             | Michael  | Suyama    | 353645,075 | 326392,105 | 458896,835 | 307516,34  | 461403,42 
7             | Robert   | King      | 484856,10  | 469048,935 | 439488,02  | 334431,125 | 564472,40 
8             | Laura    | Callahan  | 408803,575 | 360505,88 